In [ ]:
!nvidia-smi
import torch
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "Enable GPU: Runtime > Change runtime type > GPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
GDRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/tinystories_checkpoints'
os.makedirs(GDRIVE_CHECKPOINT_DIR, exist_ok=True)
os.environ['GDRIVE_PATH'] = GDRIVE_CHECKPOINT_DIR
print(f"Checkpoints will be synced to: {GDRIVE_CHECKPOINT_DIR}")

In [ ]:
REPO_URL = "https://github.com/abhishekadile/CSCI_642.git"
!git clone {REPO_URL}
%cd CSCI_642

In [ ]:
!pip install -r requirements.txt

In [ ]:
!python data/preprocess.py
# This downloads TinyStories from HuggingFace, tokenizes with GPT-2 BPE,
# chunks into 256-token sequences, saves binary tensors to data/cache/
# Takes 5-8 minutes on first run. Subsequent runs skip if cache exists.

In [ ]:
!python scripts/train.py \
  --config configs/default.yaml \
  --time_limit 3600 \
  --device cuda
# Checkpoints saved to checkpoints/ AND synced to Google Drive every 500 steps
# If Colab disconnects: re-run cells 1-5, then add --resume to this command

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

log = pd.read_csv('results/training_log.csv')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(log['step'], log['train_loss'], alpha=0.6, label='Train Loss')
val_rows = log.dropna(subset=['val_loss'])
ax.plot(val_rows['step'], val_rows['val_loss'], 'r-o', ms=4, label='Val Loss')
ax.set(xlabel='Step', ylabel='Loss', title='Training and Validation Loss')
ax.legend()

ax = axes[1]
ax.plot(val_rows['step'], val_rows['val_ppl'], 'g-o', ms=4)
ax.set(xlabel='Step', ylabel='Perplexity', title='Validation Perplexity')

plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Run this cell INSTEAD of Cell 6 if you need to resume
!python scripts/train.py \
  --config configs/default.yaml \
  --time_limit 3600 \
  --resume checkpoints/latest.pt \
  --device cuda

In [ ]:
import sys; sys.path.insert(0, '.')
import torch, yaml
from model.transformer import GPTModel
from data.tokenizer import TinyStoriesTokenizer
from inference.chat import ChatSession

config = yaml.safe_load(open('configs/default.yaml'))
tokenizer = TinyStoriesTokenizer()
model = GPTModel(config['model'])

ckpt = torch.load('checkpoints/best.pt', map_location='cuda')
model.load_state_dict(ckpt['model_state_dict'])
model = model.cuda().eval()

session = ChatSession(model, tokenizer, device='cuda')

user_input = input("You: ")
response = session.chat(user_input)
print(f"Model: {response}")